# 03 - Neural networks with PyTorch and fastai

A neural network for tabular data is a stack of layers: each layer multiplies the inputs by a weight matrix, adds a bias and applies a non-linearity. **Training** repeats four steps over the data in small batches:

1. forward pass: compute the predicted probability of a win
2. loss: how wrong was it (cross-entropy)
3. backward pass: gradient of the loss with respect to every weight
4. optimizer step: nudge each weight against its gradient by the **learning rate**

One pass over the whole training set is an **epoch**.

First you write that loop by hand in PyTorch (about 15 lines), then you let fastai do it with embeddings, normalisation and a learning-rate schedule. Everything is evaluated on matches the network has not seen.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make the `lolml` helpers importable (notebooks run from notebooks/)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from lolml import data, features as F, paths
paths.ensure_dirs()
SMOKE = bool(os.environ.get("LOLML_SMOKE"))   # LOLML_SMOKE=1 uses the small committed sample and short training budgets
SEED = 42
def save_fig(name):
    plt.tight_layout(); plt.savefig(paths.OUT / name, dpi=110); plt.show()

import torch, torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score
torch.manual_seed(SEED); np.random.seed(SEED)
print("torch", torch.__version__)
EPOCHS = 1 if SMOKE else 6

## 1. Data, split by match

The by-minute data: 10 live-safe stats of one player at a point in the game. The target is whether that player's team wins. The network standardises each column with the *training* mean and standard deviation.

In [ ]:
bym = data.load("by_minute").reset_index(drop=True)
bcols = F.BYMIN_LIVE_SAFE
tv_idx, te_idx = F.group_split(bym, 0.2, SEED)
tv = bym.iloc[tv_idx].reset_index(drop=True)
t_idx, v_idx = F.group_split(tv, 0.15, SEED + 1)
train, valid, test = tv.iloc[t_idx], tv.iloc[v_idx], bym.iloc[te_idx]
print({k: f"{len(d):,} rows / {d.match_id.nunique():,} matches" for k, d in (("train", train), ("valid", valid), ("test", test))})
mu, sd = train[bcols].mean(), train[bcols].std().replace(0, 1)
def tensors(d): return torch.tensor(((d[bcols] - mu) / sd).to_numpy(np.float32)), torch.tensor(d["winner"].to_numpy(np.float32))
(Xtr, ytr), (Xva, yva), (Xte, yte) = tensors(train), tensors(valid), tensors(test)

## 2. The training loop, written by hand

In [ ]:
model = nn.Sequential(nn.Linear(len(bcols), 200), nn.ReLU(), nn.Linear(200, 100), nn.ReLU(), nn.Linear(100, 1))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
BS = 2048
hist = []
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), BS):
        b = perm[i:i + BS]
        loss = loss_fn(model(Xtr[b]).squeeze(1), ytr[b])   # 1) forward  2) loss
        opt.zero_grad(); loss.backward()                    # 3) backward
        opt.step()                                          # 4) update the weights
    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(Xva).squeeze(1))
        vl = loss_fn(model(Xva).squeeze(1), yva).item()
    hist.append((epoch + 1, loss.item(), vl, ((p > 0.5).float() == yva).float().mean().item()))
    print(f"epoch {epoch + 1}: train loss {loss.item():.4f}  valid loss {vl:.4f}  valid accuracy {hist[-1][3]:.4f}")

In [ ]:
with torch.no_grad():
    p_te = torch.sigmoid(model(Xte).squeeze(1)).numpy()
torch_acc, torch_auc = accuracy_score(yte.numpy(), p_te > 0.5), roc_auc_score(yte.numpy(), p_te)
print(f"hand-written PyTorch MLP on unseen matches: accuracy {torch_acc:.4f}  AUC {torch_auc:.4f}")

## 3. The same network with fastai

fastai adds the parts you would otherwise write yourself: normalisation, missing-value handling, a one-cycle learning-rate schedule and metrics. `lr_find` tries learning rates from tiny to huge in one short run and suggests a good one.

In [ ]:
from fastai.tabular.all import (TabularPandas, FillMissing, Categorify, Normalize, CategoryBlock, tabular_learner,
                                RocAucBinary, accuracy, valley, load_learner)   # explicit imports: `import *` would overwrite names like F and test
tv_all = pd.concat([train, valid]).reset_index(drop=True)
splits = (list(range(len(train))), list(range(len(train), len(tv_all))))
to = TabularPandas(tv_all, procs=[FillMissing, Normalize], cont_names=bcols, y_names="winner", y_block=CategoryBlock(), splits=splits)
dls = to.dataloaders(bs=BS)
learn = tabular_learner(dls, layers=[200, 100], metrics=[accuracy, RocAucBinary()])
print([int(v) for v in learn.dls.vocab])   # the class order the model uses: [0, 1] = [loss, win]
lrs = learn.lr_find(suggest_funcs=(valley,))
print("suggested learning rate:", lrs.valley)
plt.savefig(paths.OUT / "lr_find.png", dpi=110)

In [ ]:
learn.fit_one_cycle(EPOCHS, lr_max=2e-3)
learn.recorder.plot_loss(); plt.savefig(paths.OUT / "nn_loss.png", dpi=110)

### Evaluate on the test matches

`get_preds` returns one probability per class, in the order of `learn.dls.vocab`. The original notebook printed `probs[0]` as the *winning* probability. That is the probability of **losing**: class 0 comes first. Always look the class up by name.

In [ ]:
def win_prob(df):
    dl = learn.dls.test_dl(df[bcols].assign(winner=0))
    probs, _ = learn.get_preds(dl=dl)
    return probs[:, list(learn.dls.vocab).index(1)].cpu().numpy()     # column of the class "1" = win
p_nn = win_prob(test)
nn_acc, nn_auc = accuracy_score(test["winner"], p_nn > 0.5), roc_auc_score(test["winner"], p_nn)
print(f"fastai MLP on unseen matches: accuracy {nn_acc:.4f}  AUC {nn_auc:.4f}")
row = test.iloc[[0]]
print(f"first test row: true winner={int(row.winner.iloc[0])}  P(win) = {win_prob(row)[0] * 100:.1f} %  (P(loss) = {100 - win_prob(row)[0] * 100:.1f} %)")

### A win-probability curve for one match

Replay one test match from the point of view of one player and plot P(win) at every snapshot. A **sliding window** (mean of the last 10 predictions) removes the jitter, which is what you want on a live overlay.

In [ ]:
mid = test["match_id"].iloc[0]
m = test[(test.match_id == mid) & (test.slot == 1)].sort_values("timestamp")
m = m.assign(p=win_prob(m), p_smooth=lambda d: d.p.rolling(10, min_periods=1).mean())
outcome = int(m.winner.iloc[0])
ax = m.plot(x="minute", y=["p", "p_smooth"], figsize=(7, 3.5), ylim=(0, 1), style=[":", "-"],
            title=f"P(win) during match {mid}, player 1 ({'won' if outcome else 'lost'})")
ax.axhline(0.5, color="grey", ls="--"); ax.set_ylabel("P(win)")
save_fig("nn_win_probability_curve.png")

## 4. Which inputs does the network use?

Permutation importance shuffles one input at a time and measures the drop in AUC. `lolml.importance.permutation_importance` works with any predict function. The original notebook crashed here (`sort_columns` was removed from pandas) and read a global variable instead of the learner.

In [ ]:
from lolml.importance import permutation_importance
sub = test.sample(min(len(test), 20000), random_state=SEED)
imp = permutation_importance(win_prob, sub, sub["winner"], bcols, n_repeats=3, seed=SEED)
print("baseline AUC on the subset:", round(imp.attrs["baseline_auc"], 4))
imp.set_index("feature")["importance"][::-1].plot.barh(figsize=(6, 4), title="Permutation importance (AUC drop)")
save_fig("nn_permutation_importance.png")
imp.round(4)

The game clock (`timestamp`) and the health pools lead. That makes sense: the longer the game, the more the stats say about who is ahead.

## 5. A network for the post-game data, with champion embeddings

For the post-game table the champion and position are categorical. fastai turns each category into a learned vector (an **embedding**). Champions that behave alike end up with similar vectors.

In [ ]:
perf = data.load("performance").reset_index(drop=True)
perf["win"] = perf["win"].astype(int)
pcols = F.post_game_numeric(perf)
perf[pcols] = perf[pcols].fillna(0)    # a few fields do not exist in older patches; 0 = "none" (the network cannot take NaN)
ptv, pte = F.group_split(perf, 0.2, SEED)
ptv_df, pte_df = perf.iloc[ptv].reset_index(drop=True), perf.iloc[pte]
pt, pv = F.group_split(ptv_df, 0.15, SEED + 1)
pall = pd.concat([ptv_df.iloc[pt], ptv_df.iloc[pv]]).reset_index(drop=True)
to2 = TabularPandas(pall, procs=[Categorify, Normalize], cat_names=F.DRAFT_CAT, cont_names=pcols, y_names="win", y_block=CategoryBlock(),
                    splits=(list(range(len(pt))), list(range(len(pt), len(pall)))))
learn2 = tabular_learner(to2.dataloaders(bs=1024), layers=[200, 100], metrics=[accuracy, RocAucBinary()])
learn2.fit_one_cycle(1 if SMOKE else 3, lr_max=2e-3)
dl2 = learn2.dls.test_dl(pte_df[F.DRAFT_CAT + pcols].assign(win=0))
probs2, _ = learn2.get_preds(dl=dl2)
p2 = probs2[:, list(learn2.dls.vocab).index(1)].cpu().numpy()
post_acc, post_auc = accuracy_score(pte_df["win"], p2 > 0.5), roc_auc_score(pte_df["win"], p2)
print(f"post-game fastai MLP on unseen matches: accuracy {post_acc:.4f}  AUC {post_auc:.4f}")

In [ ]:
emb = learn2.model.embeds[0].weight.detach().cpu().numpy()
print(f"championName embedding matrix: {emb.shape[0]} champions x {emb.shape[1]} learned numbers each")
print(f"position embedding matrix:     {learn2.model.embeds[1].weight.shape[0]} x {learn2.model.embeds[1].weight.shape[1]}")

Each champion is now a short vector that the network learned. Do not over-read it: the final stats (gold, damage, kills) explain almost everything, so the champion vectors carry little signal here. Embeddings shine when the category itself matters, for example in the draft model of notebook 01, or when you have many rare categories.

## 6. Export the by-minute network

The exported learner is what you load later to score new snapshots. It needs fastai and PyTorch, so the serving lab uses a smaller scikit-learn model instead.

In [ ]:
learn.export(paths.MODELS / "nn_by_minute.pkl")
reloaded = load_learner(paths.MODELS / "nn_by_minute.pkl")
dl = reloaded.dls.test_dl(test[bcols].iloc[:1000].assign(winner=0))
pr, _ = reloaded.get_preds(dl=dl)
assert np.allclose(pr[:, 1].cpu().numpy(), p_nn[:1000], atol=1e-5)
print("reloaded learner reproduces the predictions:", (paths.MODELS / "nn_by_minute.pkl").stat().st_size // 1024, "KB")
json.dump({"torch_acc": torch_acc, "torch_auc": torch_auc, "fastai_by_minute_acc": nn_acc, "fastai_by_minute_auc": nn_auc,
           "fastai_postgame_acc": post_acc, "fastai_postgame_auc": post_auc, "epochs": EPOCHS}, open(paths.OUT / "nn_results.json", "w"), indent=2)